In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import ParameterGrid

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression as LR
import optuna

from tqdm import tqdm

RANDOM_STATE = 42
TARGET_COL = "Is1Winner"

In [37]:
TRAIN_PATH = "../tourney_train_w.csv"
TEST_PATH  = "../tourney_test_w.csv"
VAL_PATH   = "../tourney_val_w.csv"

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df  = pd.read_csv(TEST_PATH, low_memory=False)
val_df   = pd.read_csv(VAL_PATH, low_memory=False)

print("train:", train_df.shape)
print("test :", test_df.shape)
print("val  :", val_df.shape)

print()
print("Target in train:", TARGET_COL in train_df.columns)
print("Target in test :", TARGET_COL in test_df.columns)
print("Target in val  :", TARGET_COL in val_df.columns)

train: (827, 175)
test : (67, 175)
val  : (67, 175)

Target in train: True
Target in test : True
Target in val  : True


In [38]:
manual_drop_cols = [
    "Season",
    "1TeamID",
    "2TeamID",
    "TeamID_2"
]

manual_drop_cols = [c for c in manual_drop_cols if c in train_df.columns]
manual_drop_cols

['Season', '1TeamID', '2TeamID', 'TeamID_2']

In [39]:
def build_feature_sets(train_df, test_df, val_df, target_col, manual_drop_cols):
    drop_cols = set(manual_drop_cols + [target_col])

    base_cols = [c for c in train_df.columns if c not in drop_cols]

    # RAW
    X_train_raw = train_df[base_cols].copy()
    X_test_raw  = test_df[base_cols].copy()
    X_val_raw   = val_df[base_cols].copy()

    # DIFF
    cols_2 = [c for c in base_cols if c.endswith("_2")]
    cols_1 = [c[:-2] for c in cols_2 if c[:-2] in base_cols]

    X_train_diff = pd.DataFrame(index=train_df.index)
    X_test_diff  = pd.DataFrame(index=test_df.index)
    X_val_diff   = pd.DataFrame(index=val_df.index)

    for c in cols_1:
        X_train_diff[f"{c}_diff"] = train_df[c] - train_df[f"{c}_2"]
        X_test_diff[f"{c}_diff"]  = test_df[c] - test_df[f"{c}_2"]
        X_val_diff[f"{c}_diff"]   = val_df[c] - val_df[f"{c}_2"]

    # RAW + DIFF
    X_train_raw_diff = pd.concat([X_train_raw, X_train_diff], axis=1)
    X_test_raw_diff  = pd.concat([X_test_raw, X_test_diff], axis=1)
    X_val_raw_diff   = pd.concat([X_val_raw, X_val_diff], axis=1)

    return {
        "raw": (X_train_raw, X_test_raw, X_val_raw),
        "diff": (X_train_diff, X_test_diff, X_val_diff),
        "raw_diff": (X_train_raw_diff, X_test_raw_diff, X_val_raw_diff)
    }
    
def clean_and_impute(X_train, X_test, X_val, missing_threshold=0.60, corr_threshold=0.995):
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_val = X_val.copy()

    # kolumny całkiem puste
    all_nan_cols = [c for c in X_train.columns if X_train[c].isna().all()]

    # zbyt duży missing
    high_missing_cols = [c for c in X_train.columns if X_train[c].isna().mean() > missing_threshold]

    # stałe
    nunique = X_train.nunique(dropna=False)
    constant_cols = nunique[nunique <= 1].index.tolist()

    drop_cols = sorted(set(all_nan_cols + high_missing_cols + constant_cols))

    X_train = X_train.drop(columns=drop_cols, errors="ignore")
    X_test  = X_test.drop(columns=drop_cols, errors="ignore")
    X_val   = X_val.drop(columns=drop_cols, errors="ignore")

    # imputacja
    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)
    X_val   = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)

    # korelacje
    if X_train.shape[1] > 1:
        corr = X_train.corr(numeric_only=True).abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        high_corr_cols = [c for c in upper.columns if any(upper[c] > corr_threshold)]
    else:
        high_corr_cols = []

    X_train = X_train.drop(columns=high_corr_cols, errors="ignore")
    X_test  = X_test.drop(columns=high_corr_cols, errors="ignore")
    X_val   = X_val.drop(columns=high_corr_cols, errors="ignore")

    meta = {
        "dropped_bad_cols": drop_cols,
        "dropped_high_corr_cols": high_corr_cols
    }

    return X_train, X_test, X_val, meta

In [40]:
def plot_corr(df, title="Correlation Matrix", figsize=(20, 16), annot=False):
    corr = df.corr(numeric_only=True)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        corr,
        mask=~mask,
        cmap="RdBu_r",
        center=0,
        vmin=-1, vmax=1,
        annot=annot,
        fmt=".2f",
        linewidths=0.5,
        square=True,
        cbar_kws={"shrink": 0.8, "label": "Correlation"},
        ax=ax
    )
    ax.set_title(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [41]:
y_train = train_df[TARGET_COL].astype(int).copy()
y_test  = test_df[TARGET_COL].astype(int).copy()
y_val   = val_df[TARGET_COL].astype(int).copy()

print(f"{len(y_train)},{len(y_test)},{len(y_val)}")


feature_sets = build_feature_sets(train_df, test_df, val_df, TARGET_COL, manual_drop_cols)

for name, (Xtr, Xte, Xva) in feature_sets.items():
    print(name, Xtr.shape, Xte.shape, Xva.shape)

827,67,67
raw (827, 170) (67, 170) (67, 170)
diff (827, 85) (67, 85) (67, 85)
raw_diff (827, 255) (67, 255) (67, 255)


In [42]:
# przetestować przy wyżuceniu większej ilości kolumn
prepared_feature_sets = {}

for feat_name, (Xtr, Xte, Xva) in feature_sets.items():
    Xtr2, Xte2, Xva2, meta = clean_and_impute(Xtr, Xte, Xva, corr_threshold=0.99)
    prepared_feature_sets[feat_name] = {
        "X_train": Xtr2,
        "X_test": Xte2,
        "X_val": Xva2,
        "meta": meta
    }
    print(f"{feat_name}: {Xtr2.shape}, {Xte2.shape}, {Xva2.shape}")

raw: (827, 160), (67, 160), (67, 160)


diff: (827, 80), (67, 80), (67, 80)
raw_diff: (827, 240), (67, 240), (67, 240)


In [43]:
def evaluate_proba(y_true, proba, threshold=0.5):
    proba = np.clip(np.asarray(proba), 1e-8, 1 - 1e-8)
    pred = (proba >= threshold).astype(int)

    return {
        "brier": brier_score_loss(y_true, proba),
    }


def print_metric_delta(train_metrics, test_metrics):
    print("TRAIN brier:", round(train_metrics["brier"], 6))
    print("TEST  brier:", round(test_metrics["brier"], 6))
    print("DELTA brier:", round(test_metrics["brier"] - train_metrics["brier"], 6))


def hard_clip_extreme(proba, high=0.85, low=0.45):
    p = np.asarray(proba).copy()
    p[p >= high] = 1.0
    p[p <= low] = 0.0
    return p


def power_sharpen(proba, alpha=1.10):
    p = np.clip(np.asarray(proba), 1e-8, 1 - 1e-8)
    num = np.power(p, alpha)
    den = num + np.power(1 - p, alpha)
    return num / den


def temperature_sharpen(proba, temperature=0.92):
    p = np.clip(np.asarray(proba), 1e-8, 1 - 1e-8)
    logits = np.log(p / (1 - p))
    logits = logits / temperature
    out = 1 / (1 + np.exp(-logits))
    return np.clip(out, 1e-8, 1 - 1e-8)

In [ ]:
def make_model(model_name, params):
    if model_name == "extratrees":
        return ExtraTreesClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        )

    if model_name == "lightgbm":
        return LGBMClassifier(
            objective="binary",
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=-1,
            **params
        )

    if model_name == "xgboost":
        return XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        )

    if model_name == "catboost":
        return CatBoostClassifier(
            loss_function="Logloss",
            verbose=False,
            random_seed=RANDOM_STATE,
            **params
        )

    if model_name == "histgb":
        return HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
            **params
        )

    if model_name == "logreg":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(random_state=RANDOM_STATE, **params))
        ])

    if model_name == "svm":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(random_state=RANDOM_STATE, **params))
        ])
    
    if model_name == "mlp":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(random_state=RANDOM_STATE, **params))
        ])

    raise ValueError(f"Nieznany model: {model_name}")


In [ ]:
# search_rows = []
# search_results = {}
# print("=" * 90)
# for feat_name, feat_data in prepared_feature_sets.items():
#     Xtr = feat_data["X_train"]
#     Xte = feat_data["X_test"]
#     Xva = feat_data["X_val"]
    
#     print(f"FEATURE SET: {feat_name}")
#     for model_name, grid in tqdm(param_spaces.items(),desc="Models tested"):
#         best_test_brier = np.inf
#         best_payload = None
        
#         for i, params in enumerate(grid, start=1):
#             model = make_model(model_name, params)
#             model.fit(Xtr, y_train)

#             train_proba = model.predict_proba(Xtr)[:, 1]
#             test_proba  = model.predict_proba(Xte)[:, 1]
#             val_proba   = model.predict_proba(Xva)[:, 1]

#             train_metrics = evaluate_proba(y_train, train_proba)
#             test_metrics  = evaluate_proba(y_test, test_proba)
#             val_metrics   = evaluate_proba(y_val, val_proba)

#             row = {
#                 "feature_set": feat_name,
#                 "model": model_name,
#                 "variant_no": i,
#                 "params": params,
#                 "train_brier": train_metrics["brier"],
#                 "test_brier": test_metrics["brier"],
#                 "val_brier": val_metrics["brier"],
#                 "brier_gap_test_minus_train": test_metrics["brier"] - train_metrics["brier"],
#                 "brier_gap_val_minus_train": val_metrics["brier"] - train_metrics["brier"]
#             }
#             search_rows.append(row)

#             if test_metrics["brier"] < best_test_brier:
#                 best_test_brier = test_metrics["brier"]
#                 best_payload = {
#                     "model": model,
#                     "params": params,
#                     "train_proba": train_proba,
#                     "test_proba": test_proba,
#                     "val_proba": val_proba,
#                     "train_metrics": train_metrics,
#                     "test_metrics": test_metrics,
#                     "val_metrics": val_metrics,
#                     "Xtr": Xtr,
#                     "Xte": Xte,
#                     "Xva": Xva
#                 }

#         search_results[(feat_name, model_name)] = best_payload
#     print("-"*60)
    
# summary = pd.DataFrame(search_rows)
# pivot = summary.loc[
#     summary.groupby(["feature_set", "model"])["test_brier"].idxmin()
# ].sort_values(["feature_set", "test_brier"])

# BOLD = "\033[1m"
# RESET = "\033[0m"

# for fs in pivot["feature_set"].unique():
#     print("=" * 90)
#     print(f"  FEATURE SET: {fs}")
#     print("=" * 90)
#     print(f"  {'Model':<15} {'Train Brier':>12} {BOLD}{'Test Brier':>12}{RESET} {'Val Brier':>12} {'Delta (T-Tr)':>14}")
#     print("-" * 90)
#     subset = pivot[pivot["feature_set"] == fs]
#     for _, r in subset.iterrows():
#         print(f"  {r['model']:<15} {r['train_brier']:>12.6f} {BOLD}{r['test_brier']:>12.6f}{RESET} {r['val_brier']:>12.6f} {r['brier_gap_test_minus_train']:>14.6f}")
#     print()

FEATURE SET: raw


Models tested: 100%|██████████| 8/8 [00:15<00:00,  1.88s/it]


------------------------------------------------------------
FEATURE SET: diff


Models tested: 100%|██████████| 8/8 [00:10<00:00,  1.31s/it]


------------------------------------------------------------
FEATURE SET: raw_diff


Models tested: 100%|██████████| 8/8 [00:20<00:00,  2.61s/it]

------------------------------------------------------------
  FEATURE SET: diff
  Model            Train Brier   Test Brier    Val Brier   Delta (T-Tr)
------------------------------------------------------------------------------------------
  logreg              0.136356     0.123319     0.109962      -0.013037
  catboost            0.053533     0.143613     0.127833       0.090080
  xgboost             0.125314     0.145132     0.121541       0.019818
  svm                 0.102601     0.148207     0.133410       0.045606
  histgb              0.033407     0.149737     0.126352       0.116330
  extratrees          0.135523     0.151604     0.140932       0.016081
  lightgbm            0.053762     0.151740     0.122039       0.097979
  mlp                 0.162361     0.164782     0.157938       0.002421

  FEATURE SET: raw
  Model            Train Brier   Test Brier    Val Brier   Delta (T-Tr)
----------------------------------------------------------------------------------------

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 80  # per model per feature set

def make_optuna_objective(model_name, Xtr, y_tr, Xte, y_te):
    
    def objective(trial):
        if model_name == "lightgbm":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 4, 31),
                "max_depth": trial.suggest_int("max_depth", 2, 6),
                "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
                "subsample": trial.suggest_float("subsample", 0.5, 0.9),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
                "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 20.0, log=True),
            }
        elif model_name == "xgboost":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 2, 5),
                "min_child_weight": trial.suggest_int("min_child_weight", 5, 30),
                "subsample": trial.suggest_float("subsample", 0.5, 0.9),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
                "gamma": trial.suggest_float("gamma", 0.0, 3.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 20.0, log=True),
            }
        elif model_name == "catboost":
            params = {
                "iterations": trial.suggest_int("iterations", 100, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "depth": trial.suggest_int("depth", 2, 6),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0),
                "random_strength": trial.suggest_float("random_strength", 0.5, 5.0),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 40),
            }
        elif model_name == "extratrees":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
                "max_depth": trial.suggest_int("max_depth", 3, 8),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 50),
                "max_features": trial.suggest_float("max_features", 0.1, 0.5),
                "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
            }
        elif model_name == "histgb":
            params = {
                "max_iter": trial.suggest_int("max_iter", 100, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 2, 6),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 50),
                "l2_regularization": trial.suggest_float("l2_regularization", 0.1, 20.0, log=True),
                "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 4, 31),
            }
        elif model_name == "logreg":
            penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet", "none"])
            params = {
                "max_iter": 3000,
            }
            if penalty == "none":
                params["penalty"] = None
                params["solver"] = "lbfgs"
            else:
                params["penalty"] = penalty
                params["C"] = trial.suggest_float("C", 1e-5, 100.0, log=True)
                params["solver"] = "saga"
                if penalty == "elasticnet":
                    params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.05, 0.95)
        elif model_name == "svm":
            params = {
                "C": trial.suggest_float("C", 0.01, 50.0, log=True),
                "kernel": "rbf",
                "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),
                "probability": True,
            }
        elif model_name == "mlp":
            n_layers = trial.suggest_int("n_layers", 1, 3)
            layers = tuple(
                trial.suggest_int(f"n_units_{i}", 16, 128) for i in range(n_layers)
            )
            params = {
                "hidden_layer_sizes": layers,
                "alpha": trial.suggest_float("alpha", 0.1, 10.0, log=True),
                "learning_rate_init": trial.suggest_float("lr", 0.0003, 0.003, log=True),
                "max_iter": 1500,
                "early_stopping": True,
            }
        else:
            raise ValueError(model_name)

        model = make_model(model_name, params)
        model.fit(Xtr, y_tr)
        proba = model.predict_proba(Xte)[:, 1]
        return brier_score_loss(y_te, proba)

    return objective


search_rows = []
search_results = {}

for feat_name, feat_data in prepared_feature_sets.items():
    Xtr = feat_data["X_train"]
    Xte = feat_data["X_test"]
    Xva = feat_data["X_val"]

    print("=" * 90)
    print(f"FEATURE SET: {feat_name}")

    for model_name in tqdm(param_spaces.keys(), desc="Optuna search"):
        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        )
        objective = make_optuna_objective(model_name, Xtr, y_train, Xte, y_test)
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

        # retrain best
        best_params = study.best_params
        
        # reconstruct MLP layers
        if model_name == "mlp":
            n_layers = best_params.pop("n_layers")
            layers = tuple(best_params.pop(f"n_units_{i}") for i in range(n_layers))
            best_params["hidden_layer_sizes"] = layers
            best_params["learning_rate_init"] = best_params.pop("lr")
            best_params["max_iter"] = 1500
            best_params["early_stopping"] = True
        if model_name == "logreg":
            best_params["solver"] = "saga"
            best_params["max_iter"] = 2000
        if model_name == "svm":
            best_params["probability"] = True
            best_params["kernel"] = "rbf"

        model = make_model(model_name, best_params)
        model.fit(Xtr, y_train)

        train_proba = model.predict_proba(Xtr)[:, 1]
        test_proba  = model.predict_proba(Xte)[:, 1]
        val_proba   = model.predict_proba(Xva)[:, 1]

        train_metrics = evaluate_proba(y_train, train_proba)
        test_metrics  = evaluate_proba(y_test, test_proba)
        val_metrics   = evaluate_proba(y_val, val_proba)

        search_rows.append({
            "feature_set": feat_name,
            "model": model_name,
            "params": best_params,
            "train_brier": train_metrics["brier"],
            "test_brier": test_metrics["brier"],
            "val_brier": val_metrics["brier"],
            "brier_gap_test_minus_train": test_metrics["brier"] - train_metrics["brier"],
            "brier_gap_val_minus_train": val_metrics["brier"] - train_metrics["brier"],
            "n_trials": len(study.trials),
        })

        search_results[(feat_name, model_name)] = {
            "model": model,
            "params": best_params,
            "train_proba": train_proba,
            "test_proba": test_proba,
            "val_proba": val_proba,
            "train_metrics": train_metrics,
            "test_metrics": test_metrics,
            "val_metrics": val_metrics,
            "Xtr": Xtr,
            "Xte": Xte,
            "Xva": Xva,
        }

    print("-" * 60)

summary = pd.DataFrame(search_rows)

BOLD = "\033[1m"
RESET = "\033[0m"

for fs in summary["feature_set"].unique():
    print("=" * 90)
    print(f"  FEATURE SET: {fs}")
    print("=" * 90)
    print(f"  {'Model':<15} {'Train Brier':>12} {BOLD}{'Test Brier':>12}{RESET} {'Val Brier':>12} {'Delta (T-Tr)':>14}")
    print("-" * 90)
    subset = summary[summary["feature_set"] == fs].sort_values("test_brier")
    for _, r in subset.iterrows():
        print(f"  {r['model']:<15} {r['train_brier']:>12.6f} {BOLD}{r['test_brier']:>12.6f}{RESET} {r['val_brier']:>12.6f} {r['brier_gap_test_minus_train']:>14.6f}")
    print()

FEATURE SET: raw


Optuna search:   0%|          | 0/8 [00:00<?, ?it/s]

Optuna search: 100%|██████████| 8/8 [05:52<00:00, 44.08s/it]


------------------------------------------------------------
FEATURE SET: diff


Optuna search: 100%|██████████| 8/8 [04:38<00:00, 34.83s/it]


------------------------------------------------------------
FEATURE SET: raw_diff


Optuna search: 100%|██████████| 8/8 [08:43<00:00, 65.43s/it]

------------------------------------------------------------
  FEATURE SET: raw
  Model            Train Brier   Test Brier    Val Brier   Delta (T-Tr)
------------------------------------------------------------------------------------------
  logreg              0.127393     0.127266     0.108966      -0.000128
  lightgbm            0.109942     0.136373     0.115643       0.026431
  histgb              0.123765     0.136690     0.119083       0.012926
  mlp                 0.109537     0.138425     0.143115       0.028888
  xgboost             0.108349     0.138761     0.118245       0.030412
  extratrees          0.121116     0.139387     0.126369       0.018271
  catboost            0.106831     0.142009     0.118272       0.035178
  svm                 0.127590     0.146685     0.125841       0.019095

  FEATURE SET: diff
  Model            Train Brier   Test Brier    Val Brier   Delta (T-Tr)
----------------------------------------------------------------------------------------

In [121]:
clip_configs = [
    ("no_clip", None, None),
    ("clip_01_99", 0.01, 0.99),
    ("clip_05_95", 0.05, 0.95),
    ("clip_10_90", 0.10, 0.90),
    ("clip_15_85", 0.15, 0.85),
    ("clip_20_80", 0.20, 0.80),
    ("clip_25_75", 0.25, 0.75),
    ("clip_30_70", 0.30, 0.70),
    ("clip_35_65", 0.35, 0.65),
    ("clip_40_60", 0.40, 0.60),
]
power_alphas = np.round(np.delete(np.arange(0.5,2,step=0.05),10),5)

In [122]:
postprocess_rows = []
print("=" * 90)
for feat_name, feat_data in prepared_feature_sets.items():
    print(f"FEATURE SET: {feat_name}")
    
    for model_name in tqdm(param_spaces.keys(),desc="POSTPROCESSING TESTED"):
        payload = search_results[(feat_name, model_name)]
        best_model = payload["model"]

        train_raw = payload["train_proba"]
        test_raw  = payload["test_proba"]
        val_raw   = payload["val_proba"]

        def add_row(transform, param, train_p, test_p, val_p):
            train_m = evaluate_proba(y_train, train_p)
            test_m  = evaluate_proba(y_test, test_p)
            val_m   = evaluate_proba(y_val, val_p)
            postprocess_rows.append({
                "feature_set": feat_name,
                "model": model_name,
                "transform": transform,
                "param": param,
                "train_brier": train_m["brier"],
                "test_brier": test_m["brier"],
                "val_brier": val_m["brier"],
                "delta": test_m["brier"] - train_m["brier"],
            })

        # raw
        add_row("raw", "-", train_raw, test_raw, val_raw)

        # clip
        for clip_name, lo, hi in clip_configs:
            if lo is not None:
                add_row("clip", f"{lo}-{hi}",
                        np.clip(train_raw, lo, hi),
                        np.clip(test_raw, lo, hi),
                        np.clip(val_raw, lo, hi))

        # power
        for alpha in power_alphas:
            if alpha != 1.0:
                add_row("power", alpha,
                        power_sharpen(train_raw, alpha=alpha),
                        power_sharpen(test_raw, alpha=alpha),
                        power_sharpen(val_raw, alpha=alpha))

        # temperature
        for temp in temperatures:
            if temp != 1.0:
                add_row("temperature", temp,
                        temperature_sharpen(train_raw, temperature=temp),
                        temperature_sharpen(test_raw, temperature=temp),
                        temperature_sharpen(val_raw, temperature=temp))

    print("-"*90)

pp_df = pd.DataFrame(postprocess_rows)

# ===== BEST POSTPROCESS PER (FEATURE_SET, MODEL) =====
print(f"\n{'═' * 110}")
print("BEST POSTPROCESS PER FEATURE SET + MODEL (by test_brier)")
print(f"{'═' * 110}")
print(f"  {'FeatureSet':<12} {'Model':<15} {'Transform':<15} {'Param':<10} {'Train':>10} {'Test':>10} {'Val':>10} {'Delta':>10}")
print(f"  {'─' * 100}")

best_pp = pp_df.loc[pp_df.groupby(["feature_set", "model"])["test_brier"].idxmin()]
best_pp = best_pp.sort_values(["test_brier"])

for _, r in best_pp.iterrows():
    print(f"  {r['feature_set']:<12} {r['model']:<15} {r['transform']:<15} {str(r['param']):<10} {r['train_brier']:>10.6f} {r['test_brier']:>10.6f} {r['val_brier']:>10.6f} {r['delta']:>10.6f}")

# ===== TOP 20 OVERALL =====
print(f"\n{'═' * 110}")
print("TOP 20 OVERALL")
print(f"{'═' * 110}")
print(f"  {'FeatureSet':<12} {'Model':<15} {'Transform':<15} {'Param':<10} {'Train':>10} {'Test':>10} {'Val':>10} {'Delta':>10}")
print(f"  {'─' * 100}")

for _, r in pp_df.sort_values("test_brier").head(20).iterrows():
    print(f"  {r['feature_set']:<12} {r['model']:<15} {r['transform']:<15} {str(r['param']):<10} {r['train_brier']:>10.6f} {r['test_brier']:>10.6f} {r['val_brier']:>10.6f} {r['delta']:>10.6f}")

FEATURE SET: raw


POSTPROCESSING TESTED: 100%|██████████| 8/8 [00:00<00:00, 20.57it/s]


------------------------------------------------------------------------------------------
FEATURE SET: diff


POSTPROCESSING TESTED: 100%|██████████| 8/8 [00:00<00:00, 26.67it/s]


------------------------------------------------------------------------------------------
FEATURE SET: raw_diff


POSTPROCESSING TESTED: 100%|██████████| 8/8 [00:00<00:00, 23.92it/s]

------------------------------------------------------------------------------------------

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
BEST POSTPROCESS PER FEATURE SET + MODEL (by test_brier)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  FeatureSet   Model           Transform       Param           Train       Test        Val      Delta
  ────────────────────────────────────────────────────────────────────────────────────────────────────
  diff         logreg          power           1.6          0.140077   0.117702   0.101608  -0.022375
  raw_diff     logreg          power           1.35         0.129284   0.122903   0.105123  -0.006381
  raw_diff     extratrees      power           1.95         0.124983   0.124787   0.106835  -0.000197
  raw          logreg          power           1.25         0.127256   0.125847   0.105827  -0.001408
  diff         x

In [123]:
TOP_K = 6

best_pp = pp_df.loc[pp_df.groupby(["feature_set", "model"])["test_brier"].idxmin()]
best_pp = best_pp.sort_values("test_brier").head(TOP_K)

print("Top models for ensemble:")
for _, r in best_pp.iterrows():
    print(f"  {r['feature_set']:<12} {r['model']:<15} {r['transform']:<12} {str(r['param']):<10} test={r['test_brier']:.6f}")

Top models for ensemble:
  diff         logreg          power        1.6        test=0.117702
  raw_diff     logreg          power        1.35       test=0.122903
  raw_diff     extratrees      power        1.95       test=0.124787
  raw          logreg          power        1.25       test=0.125847
  diff         xgboost         power        1.4        test=0.126637
  raw          extratrees      power        1.95       test=0.127062


In [124]:
def apply_postprocess(proba, transform, param):
    if transform == "raw":
        return proba
    if transform == "clip":
        lo, hi = map(float, param.split("-"))
        return np.clip(proba, lo, hi)
    if transform == "power":
        return power_sharpen(proba, alpha=float(param))
    if transform == "temperature":
        return temperature_sharpen(proba, temperature=float(param))
    return proba

ensemble_models = []

for _, r in best_pp.iterrows():
    feat_name = r["feature_set"]
    model_name = r["model"]
    transform = r["transform"]
    param = r["param"]

    # weź wytrenowany model z search_results
    payload = search_results[(feat_name, model_name)]
    model = payload["model"]
    Xtr = payload["Xtr"]
    Xte = payload["Xte"]
    Xva = payload["Xva"]

    # surowe predykcje
    train_raw = model.predict_proba(Xtr)[:, 1]
    test_raw  = model.predict_proba(Xte)[:, 1]
    val_raw   = model.predict_proba(Xva)[:, 1]

    # postprocessing
    train_pp = apply_postprocess(train_raw, transform, param)
    test_pp  = apply_postprocess(test_raw, transform, param)
    val_pp   = apply_postprocess(val_raw, transform, param)

    test_brier = brier_score_loss(y_test, test_pp)

    ensemble_models.append({
        "feat_name": feat_name,
        "model_name": model_name,
        "transform": transform,
        "param": param,
        "model": model,
        "train_pp": train_pp,
        "test_pp": test_pp,
        "val_pp": val_pp,
        "test_brier": test_brier,
    })

In [125]:
weights = np.array([1.0 / m["test_brier"] for m in ensemble_models])
weights /= weights.sum()

print(f"\nWagi ensemble ({TOP_K} modeli):")
for m, w in zip(ensemble_models, weights):
    print(f"  {m['feat_name']:<12} {m['model_name']:<15} {m['transform']:<12} w={w:.4f}")

train_ensemble = sum(w * m["train_pp"] for w, m in zip(weights, ensemble_models))
test_ensemble  = sum(w * m["test_pp"]  for w, m in zip(weights, ensemble_models))
val_ensemble   = sum(w * m["val_pp"]   for w, m in zip(weights, ensemble_models))

print(f"\n{'═' * 70}")
print("ENSEMBLE (ważona średnia z postprocessingiem per model)")
print(f"{'═' * 70}")
ens_train = evaluate_proba(y_train, train_ensemble)
ens_test  = evaluate_proba(y_test, test_ensemble)
ens_val   = evaluate_proba(y_val, val_ensemble)
print(f"  Train brier: {ens_train['brier']:.6f}")
print(f"  Test  brier: {ens_test['brier']:.6f}")
print(f"  Val   brier: {ens_val['brier']:.6f}")
print(f"  Delta:       {ens_test['brier'] - ens_train['brier']:.6f}")


Wagi ensemble (6 modeli):
  diff         logreg          power        w=0.1757
  raw_diff     logreg          power        w=0.1683
  raw_diff     extratrees      power        w=0.1657
  raw          logreg          power        w=0.1643
  diff         xgboost         power        w=0.1633
  raw          extratrees      power        w=0.1627

══════════════════════════════════════════════════════════════════════
ENSEMBLE (ważona średnia z postprocessingiem per model)
══════════════════════════════════════════════════════════════════════
  Train brier: 0.123032
  Test  brier: 0.118833
  Val   brier: 0.102859
  Delta:       -0.004199


In [126]:
ens_pp_rows = []
# raw ensemble
ens_pp_rows.append({
    "transform": "raw", 
    "param": "-",
    "train_brier": ens_train["brier"],
    "test_brier": ens_test["brier"],
    "val_brier": ens_val["brier"],
    "delta": ens_test["brier"] - ens_train["brier"]
})

for clip_name, lo, hi in clip_configs:
    if lo is not None:
        t = evaluate_proba(y_train, np.clip(train_ensemble, lo, hi))
        e = evaluate_proba(y_test, np.clip(test_ensemble, lo, hi))
        v = evaluate_proba(y_val, np.clip(val_ensemble, lo, hi))
        ens_pp_rows.append({"transform": "clip", "param": f"{lo}-{hi}", "train_brier": t["brier"], "test_brier": e["brier"], "val_brier": v["brier"], "delta": e["brier"] - t["brier"]})

for alpha in power_alphas:
    if alpha != 1.0:
        t = evaluate_proba(y_train, power_sharpen(train_ensemble, alpha=alpha))
        e = evaluate_proba(y_test, power_sharpen(test_ensemble, alpha=alpha))
        v = evaluate_proba(y_val, power_sharpen(val_ensemble, alpha=alpha))
        ens_pp_rows.append({"transform": "power", "param": alpha, "train_brier": t["brier"], "test_brier": e["brier"], "val_brier": v["brier"], "delta": e["brier"] - t["brier"]})

for temp in temperatures:
    if temp != 1.0:
        t = evaluate_proba(y_train, temperature_sharpen(train_ensemble, temperature=temp))
        e = evaluate_proba(y_test, temperature_sharpen(test_ensemble, temperature=temp))
        v = evaluate_proba(y_val, temperature_sharpen(val_ensemble, temperature=temp))
        ens_pp_rows.append({"transform": "temperature", "param": temp, "train_brier": t["brier"], "test_brier": e["brier"], "val_brier": v["brier"], "delta": e["brier"] - t["brier"]})

ens_pp_df = pd.DataFrame(ens_pp_rows).sort_values("test_brier")

print(f"\n{'═' * 90}")
print("POSTPROCESSING FOR ENSEMBLE")
print(f"{'═' * 90}")
print(f"  {'Transform':<15} {'Param':<10} {'Train':>10} {'Test':>10} {'Val':>10} {'Delta':>10}")
print(f"  {'─' * 65}")
for _, r in ens_pp_df.iterrows():
    print(f"  {r['transform']:<15} {str(r['param']):<10} {r['train_brier']:>10.6f} {r['test_brier']:>10.6f} {r['val_brier']:>10.6f} {r['delta']:>10.6f}")


══════════════════════════════════════════════════════════════════════════════════════════
POSTPROCESSING FOR ENSEMBLE
══════════════════════════════════════════════════════════════════════════════════════════
  Transform       Param           Train       Test        Val      Delta
  ─────────────────────────────────────────────────────────────────
  power           1.15         0.124285   0.118350   0.101296  -0.005935
  power           1.2          0.124843   0.118388   0.100991  -0.006455
  power           1.1          0.123786   0.118402   0.101698  -0.005385
  power           1.25         0.125448   0.118501   0.100770  -0.006947
  power           1.05         0.123362   0.118557   0.102213  -0.004806
  power           1.3          0.126089   0.118680   0.100622  -0.007409
  raw             -            0.123032   0.118833   0.102859  -0.004199
  clip            0.01-0.99    0.123038   0.118836   0.102863  -0.004202
  power           1.35         0.126757   0.118914   0.100538  -

In [127]:
best_single = pp_df.sort_values("test_brier").iloc[0]
best_ens = ens_pp_df.sort_values("test_brier").iloc[0]

print(f"\n{'═' * 70}")
print("COMPARISON")
print(f"{'═' * 70}")
print(f"BEST SINGLE: {best_single['feature_set']}/{best_single['model']}/{best_single['transform']}({best_single['param']})")
print(f"    test_brier = {best_single['test_brier']:.6f}  delta = {best_single['delta']:.6f}")
print(f"BEST ENSEMBLE + POSTPROCESSING: {best_ens['transform']}({best_ens['param']})")
print(f"    test_brier = {best_ens['test_brier']:.6f}  delta = {best_ens['delta']:.6f}")


══════════════════════════════════════════════════════════════════════
COMPARISON
══════════════════════════════════════════════════════════════════════
BEST SINGLE: diff/logreg/power(1.6)
    test_brier = 0.117702  delta = -0.022375
BEST ENSEMBLE + POSTPROCESSING: power(1.15)
    test_brier = 0.118350  delta = -0.005935


In [128]:
calibrated_models_pp = []    # postprocessing -> kalibracja
calibrated_models_raw = []   # raw -> kalibracja

for m in ensemble_models:
    feat_name = m["feat_name"]
    model_name = m["model_name"]

    payload = search_results[(feat_name, model_name)]
    train_raw = payload["train_proba"]
    test_raw  = payload["test_proba"]
    val_raw   = payload["val_proba"]

    # === WARIANT 1: postprocessing -> kalibracja ===
    train_pp = apply_postprocess(train_raw, m["transform"], m["param"])
    test_pp  = apply_postprocess(test_raw, m["transform"], m["param"])
    val_pp   = apply_postprocess(val_raw, m["transform"], m["param"])

    cal_pp = LogisticRegression(C=1.0, max_iter=1000)
    cal_pp.fit(test_pp.reshape(-1, 1), y_test)

    train_pp_cal = cal_pp.predict_proba(train_pp.reshape(-1, 1))[:, 1]
    test_pp_cal  = cal_pp.predict_proba(test_pp.reshape(-1, 1))[:, 1]
    val_pp_cal   = cal_pp.predict_proba(val_pp.reshape(-1, 1))[:, 1]

    calibrated_models_pp.append({
        **m,
        "train_cal": train_pp_cal,
        "test_cal": test_pp_cal,
        "val_cal": val_pp_cal,
        "val_brier_cal": brier_score_loss(y_val, val_pp_cal),
        "calibrator": cal_pp,
    })

    # === WARIANT 2: raw -> kalibracja ===
    cal_raw = LogisticRegression(penalty=None, max_iter=1000)
    cal_raw.fit(test_raw.reshape(-1, 1), y_test)

    train_raw_cal = cal_raw.predict_proba(train_raw.reshape(-1, 1))[:, 1]
    test_raw_cal  = cal_raw.predict_proba(test_raw.reshape(-1, 1))[:, 1]
    val_raw_cal   = cal_raw.predict_proba(val_raw.reshape(-1, 1))[:, 1]

    calibrated_models_raw.append({
        **m,
        "train_cal": train_raw_cal,
        "test_cal": test_raw_cal,
        "val_cal": val_raw_cal,
        "val_brier_cal": brier_score_loss(y_val, val_raw_cal),
        "calibrator": cal_raw,
    })

    print(f"  {feat_name:<12} {model_name:<15} "
          f"val: pp->cal={brier_score_loss(y_val, val_pp_cal):.6f}  "
          f"raw->cal={brier_score_loss(y_val, val_raw_cal):.6f}  "
          f"pp_only={brier_score_loss(y_val, val_pp):.6f}")


# === ENSEMBLE: postprocessing -> kalibracja ===
w_pp = np.array([1.0 / m["val_brier_cal"] for m in calibrated_models_pp])
w_pp /= w_pp.sum()

train_ens_pp = sum(w * m["train_cal"] for w, m in zip(w_pp, calibrated_models_pp))
test_ens_pp  = sum(w * m["test_cal"]  for w, m in zip(w_pp, calibrated_models_pp))
val_ens_pp   = sum(w * m["val_cal"]   for w, m in zip(w_pp, calibrated_models_pp))

# === ENSEMBLE: raw -> kalibracja ===
w_raw = np.array([1.0 / m["val_brier_cal"] for m in calibrated_models_raw])
w_raw /= w_raw.sum()

train_ens_raw = sum(w * m["train_cal"] for w, m in zip(w_raw, calibrated_models_raw))
test_ens_raw  = sum(w * m["test_cal"]  for w, m in zip(w_raw, calibrated_models_raw))
val_ens_raw   = sum(w * m["val_cal"]   for w, m in zip(w_raw, calibrated_models_raw))

# === POROWNANIE ===
train_ens_pp_only = sum(w * m["train_pp"] for w, m in zip(weights, ensemble_models))
test_ens_pp_only  = sum(w * m["test_pp"]  for w, m in zip(weights, ensemble_models))
val_ens_pp_only   = sum(w * m["val_pp"]   for w, m in zip(weights, ensemble_models))

print(f"\n{'═' * 80}")
print("POROWNANIE ENSEMBLE'I Z KALIBRACJA")
print(f"{'═' * 80}")
print(f"  {'Wariant':<30} {'Train':>10} {'Test (in-s)':>12} {'Val (oos)':>12}")
print(f"  {'─' * 70}")

for label, tr, te, va in [
    ("ensemble + pp + kalibracja",    train_ens_pp,      test_ens_pp,      val_ens_pp),
    ("ensemble + raw + kalibracja",   train_ens_raw,     test_ens_raw,     val_ens_raw),
    ("ensemble + pp", train_ens_pp_only,  test_ens_pp_only, val_ens_pp_only),
]:
    metrics_tr = evaluate_proba(y_train, tr)
    metrics_te = evaluate_proba(y_test, te)
    metrics_va = evaluate_proba(y_val, va)
    print(f"  {label:<30} {metrics_tr['brier']:>10.6f} {metrics_te['brier']:>12.6f} {metrics_va['brier']:>12.6f}")

  diff         logreg          val: pp->cal=0.122642  raw->cal=0.102233  pp_only=0.101608
  raw_diff     logreg          val: pp->cal=0.124925  raw->cal=0.103071  pp_only=0.105123
  raw_diff     extratrees      val: pp->cal=0.127840  raw->cal=0.103067  pp_only=0.106835
  raw          logreg          val: pp->cal=0.124414  raw->cal=0.101551  pp_only=0.105827
  diff         xgboost         val: pp->cal=0.138117  raw->cal=0.122466  pp_only=0.120893
  raw          extratrees      val: pp->cal=0.129199  raw->cal=0.105810  pp_only=0.108419

════════════════════════════════════════════════════════════════════════════════
POROWNANIE ENSEMBLE'I Z KALIBRACJA
════════════════════════════════════════════════════════════════════════════════
  Wariant                             Train  Test (in-s)    Val (oos)
  ──────────────────────────────────────────────────────────────────────
  ensemble + pp + kalibracja       0.138046     0.135720     0.125454
  ensemble + raw + kalibracja      0.126027     0

In [129]:
best_single = pp_df.sort_values("test_brier").iloc[0]
best_ens_raw = ens_pp_df.sort_values("test_brier").iloc[0]

val_brier_pp_cal  = evaluate_proba(y_val, val_ens_pp)["brier"]
val_brier_raw_cal = evaluate_proba(y_val, val_ens_raw)["brier"]
val_brier_pp_only = evaluate_proba(y_val, val_ens_pp_only)["brier"]

print(f"\n{'═' * 70}")
print("POROWNANIE FINALNE")
print(f"{'═' * 70}")
print(f"  {'Metoda':<40} {'Val Brier (oos)':>14}")
print(f"  {'─' * 56}")
print(f"  {'Best single model':<40} {best_single['val_brier']:>14.6f}")
print(f"  {'Ensemble + PP':<40} {val_brier_pp_only:>14.6f}")
print(f"  {'Ensemble + PP + kalibracja':<40} {val_brier_pp_cal:>14.6f}")
print(f"  {'Ensemble + raw + kalibracja':<40} {val_brier_raw_cal:>14.6f}")


══════════════════════════════════════════════════════════════════════
POROWNANIE FINALNE
══════════════════════════════════════════════════════════════════════
  Metoda                                   Val Brier (oos)
  ────────────────────────────────────────────────────────
  Best single model                              0.101608
  Ensemble + PP                                  0.102859
  Ensemble + PP + kalibracja                     0.125454
  Ensemble + raw + kalibracja                    0.100051
